# Práctica de detección de objetos: Lettuce Pallets
**Modelos:** Cascade R-CNN (Detectron2) y SSD300-VGG16 (Torchvision)

Este cuaderno implementa una plantilla reproducible para: inspección del dataset, entrenamiento baseline, evaluación y una mejora experimental. Ejecuta las celdas en orden. Se recomienda Google Colab con GPU.

**Dataset:** 1.510 imágenes, 5 clases. El dataset original está publicado por Francesco en Hugging Face y el guion indica utilizar las particiones train/validation/test. Las clases conocidas son `Ready`, `empty_pod`, `germination`, `pod`, `young`.

> Importante: los nombres exactos de los JSON, las rutas y los `category_id` se detectan/validan al inspeccionar el archivo descargado. No se debe evaluar sobre test durante el ajuste de hiperparámetros.

## 0. Entorno

Detectron2 requiere una instalación compatible con la versión de PyTorch/CUDA de la máquina. En Colab, si la instalación falla por incompatibilidad, reinicia el runtime y usa una imagen de Colab con PyTorch/CUDA compatible. Ejecuta esta celda una sola vez.

In [1]:
# En Colab:
!pip -q install pycocotools torchmetrics

# Detectron2: seguir las instrucciones oficiales para la versión de PyTorch/CUDA instalada.
# En algunos runtimes Colab puede ser necesario compilar desde fuente:
# !python -m pip install 'git+https://github.com/facebookresearch/detectron2.git'


[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: C:\Users\Acer\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [1]:
import os, json, random, time, copy, glob, zipfile, tarfile, subprocess, sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision.transforms import functional as TF
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
from google.colab import drive
import tarfile

# Mount Google Drive
drive.mount('/content/drive')

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__, "| Torchvision:", torchvision.__version__, "| Device:", DEVICE)

ModuleNotFoundError: No module named 'google.colab'

## 1. Descargar y descomprimir el dataset

Puedes subir `dataset.tar.gz` a Colab o descargar el archivo original del repositorio indicado en el guion. La extracción se realiza en una carpeta local.

In [ ]:
DATA_ARCHIVE = Path("dataset.tar.gz")
ZIP_PATH = "/content/drive/MyDrive/material.zip"
DATA_ROOT = Path("dataset")


with tarfile.open(ZIP_PATH, 'r:gz') as tar:
    tar.extractall(path='dataset')

Dataset extraído en: C:\Users\Acer\OneDrive - Universidade de Santiago de Compostela\Escritorio\vision\practicas\practica6\dataset


In [ ]:
# Opción A: si has subido dataset.tar.gz a Colab, déjalo en el directorio actual.
# Opción B: descarga el archivo original del repositorio:
if not DATA_ARCHIVE.exists():
    try:
        from huggingface_hub import hf_hub_download
        downloaded = hf_hub_download(
            repo_id="Francesco/lettuce-pallets",
            repo_type="dataset",
            filename="dataset.tar.gz"
        )
        DATA_ARCHIVE = Path(downloaded)
    except Exception as e:
        raise FileNotFoundError(
            "No encuentro dataset.tar.gz. Súbelo a Colab o descarga el archivo "
            "desde https://huggingface.co/datasets/Francesco/lettuce-pallets"
        ) from e

DATA_ROOT.mkdir(exist_ok=True)
if not any(DATA_ROOT.iterdir()):
    with tarfile.open(DATA_ARCHIVE, "r:gz") as tar:
        # Evita rutas absolutas o traversal al extraer archivos.
        base = DATA_ROOT.resolve()
        for member in tar.getmembers():
            target = (DATA_ROOT / member.name).resolve()
            if base not in target.parents and target != base:
                raise RuntimeError(f"Ruta insegura en el tar: {member.name}")
        tar.extractall(DATA_ROOT)

print("Dataset extraído en:", DATA_ROOT.resolve())

## 2. Localizar anotaciones COCO y particiones

El guion especifica particiones train/validation/test. Esta celda busca JSON COCO dentro de la extracción. Si no detecta automáticamente las tres particiones, ajusta manualmente `ANN_FILES` e `IMAGE_DIRS` después de revisar el listado.

In [4]:
json_files = list(DATA_ROOT.rglob("*.json"))
print("JSON encontrados:")
for p in json_files[:50]: print(" -", p)

def split_token(path):
    s = str(path).lower()
    if "train" in s: return "train"
    if "valid" in s or "val" in s: return "val"
    if "test" in s: return "test"
    return None

ANN_FILES = {}
for split in ("train", "val", "test"):
    candidates = [p for p in json_files if split_token(p) == split and
                  ("annotation" in str(p).lower() or "instance" in p.name.lower() or "coco" in p.name.lower())]
    if not candidates:
        candidates = [p for p in json_files if split_token(p) == split]
    if candidates:
        ANN_FILES[split] = candidates[0]

# Directorios de imágenes: busca carpetas cuyo nombre sea images y cuyo camino indique el split.
IMAGE_DIRS = {}
for split in ("train", "val", "test"):
    candidates = [p for p in DATA_ROOT.rglob("*") if p.is_dir() and p.name.lower() in ("images", "imgs")
                  and split_token(p.parent) == split]
    if candidates: IMAGE_DIRS[split] = candidates[0]

print("ANN_FILES:", ANN_FILES)
print("IMAGE_DIRS:", IMAGE_DIRS)
assert "train" in ANN_FILES and "val" in ANN_FILES, (
    "No se han detectado train y validation. Edita ANN_FILES manualmente usando el listado anterior."
)
assert "train" in IMAGE_DIRS and "val" in IMAGE_DIRS, (
    "No se han detectado carpetas de imágenes. Edita IMAGE_DIRS manualmente."
)

JSON encontrados:
 - dataset\home\zuppif\Documents\Work\RoboFlow\ODinW-RF100-challenge\rf100\lettuce-pallets\test\_annotations.coco.json
 - dataset\home\zuppif\Documents\Work\RoboFlow\ODinW-RF100-challenge\rf100\lettuce-pallets\train\_annotations.coco.json
 - dataset\home\zuppif\Documents\Work\RoboFlow\ODinW-RF100-challenge\rf100\lettuce-pallets\valid\_annotations.coco.json
ANN_FILES: {'train': WindowsPath('dataset/home/zuppif/Documents/Work/RoboFlow/ODinW-RF100-challenge/rf100/lettuce-pallets/train/_annotations.coco.json'), 'val': WindowsPath('dataset/home/zuppif/Documents/Work/RoboFlow/ODinW-RF100-challenge/rf100/lettuce-pallets/valid/_annotations.coco.json'), 'test': WindowsPath('dataset/home/zuppif/Documents/Work/RoboFlow/ODinW-RF100-challenge/rf100/lettuce-pallets/test/_annotations.coco.json')}
IMAGE_DIRS: {}


AssertionError: No se han detectado carpetas de imágenes. Edita IMAGE_DIRS manualmente.

## 3. Comprobar clases, distribución y ejemplos

Las cajas COCO usan `[x_min, y_min, width, height]`. En Detectron2 y Torchvision se convertirán a `[x_min, y_min, x_max, y_max]`.

In [ ]:
coco_train = COCO(str(ANN_FILES["train"]))
cats = sorted(coco_train.loadCats(coco_train.getCatIds()), key=lambda x: x["id"])
print("Categorías COCO:", [(c["id"], c["name"]) for c in cats])
print("Número de imágenes train:", len(coco_train.getImgIds()))
print("Número de anotaciones train:", len(coco_train.getAnnIds()))

cat_id_to_name = {c["id"]: c["name"] for c in cats}
cat_ids = [c["id"] for c in cats]
NUM_CLASSES = len(cat_ids)
assert NUM_CLASSES == 5, f"Se esperaban 5 clases según el guion; se encontraron {NUM_CLASSES}."
# Mapear IDs COCO arbitrarios a índices contiguos 0..C-1 (Torchvision).
CAT_TO_CONTIG = {cid: i for i, cid in enumerate(cat_ids)}
CLASS_NAMES = [cat_id_to_name[cid] for cid in cat_ids]
print("Orden de clases usado:", CLASS_NAMES)

def count_categories(coco):
    counts = {cat_id_to_name[cid]: 0 for cid in cat_ids}
    for ann in coco.dataset["annotations"]:
        if ann["category_id"] in cat_id_to_name:
            counts[cat_id_to_name[ann["category_id"]]] += 1
    return counts

split_counts = {}
for split, ann_path in ANN_FILES.items():
    c = COCO(str(ann_path))
    split_counts[split] = count_categories(c)
display(pd.DataFrame(split_counts).fillna(0).astype(int))

In [ ]:
# Visualizar imágenes con cajas reales.
def show_coco_samples(coco, image_dir, n=4):
    ids = random.sample(coco.getImgIds(), min(n, len(coco.getImgIds())))
    fig, axes = plt.subplots(1, len(ids), figsize=(5*len(ids), 5))
    if len(ids) == 1: axes = [axes]
    for ax, iid in zip(axes, ids):
        info = coco.loadImgs([iid])[0]
        img_path = Path(image_dir) / info["file_name"]
        if not img_path.exists():
            # Algunas exportaciones guardan basename en un subdirectorio.
            matches = list(Path(image_dir).rglob(Path(info["file_name"]).name))
            if not matches: raise FileNotFoundError(img_path)
            img_path = matches[0]
        im = Image.open(img_path).convert("RGB")
        ax.imshow(im); ax.axis("off")
        for a in coco.loadAnns(coco.getAnnIds(imgIds=[iid])):
            x,y,w,h = a["bbox"]
            ax.add_patch(plt.Rectangle((x,y),w,h,fill=False,edgecolor="red",linewidth=1.5))
            ax.text(x,y,cat_id_to_name[a["category_id"]],color="yellow",fontsize=8,
                    bbox=dict(facecolor="black",alpha=.5))
        ax.set_title(info["file_name"])
    plt.tight_layout()

show_coco_samples(coco_train, IMAGE_DIRS["train"])

## 4. Dataset común para Torchvision SSD300

Torchvision espera un tensor de imagen y un diccionario con `boxes` en XYXY y `labels` en índices contiguos desde 1 (0 está reservado para background). Se conservan las particiones originales.

In [ ]:
class CocoDetectionTorchvision(Dataset):
    def __init__(self, ann_file, image_dir, train=False):
        self.coco = COCO(str(ann_file))
        self.image_dir = Path(image_dir)
        self.ids = sorted(self.coco.getImgIds())
        self.train = train
    def __len__(self): return len(self.ids)
    def _image_path(self, file_name):
        p = self.image_dir / file_name
        if p.exists(): return p
        matches = list(self.image_dir.rglob(Path(file_name).name))
        if matches: return matches[0]
        raise FileNotFoundError(f"No se encuentra {file_name} en {self.image_dir}")
    def __getitem__(self, idx):
        image_id = self.ids[idx]
        info = self.coco.loadImgs([image_id])[0]
        image = Image.open(self._image_path(info["file_name"])).convert("RGB")
        anns = self.coco.loadAnns(self.coco.getAnnIds(imgIds=[image_id]))
        boxes, labels, areas, crowds = [], [], [], []
        W,H = image.size
        for a in anns:
            x,y,w,h = a["bbox"]
            x1,y1 = max(0,x), max(0,y)
            x2,y2 = min(W,x+w), min(H,y+h)
            if x2 <= x1 or y2 <= y1: continue
            boxes.append([x1,y1,x2,y2])
            labels.append(CAT_TO_CONTIG[a["category_id"]] + 1)
            areas.append((x2-x1)*(y2-y1)); crowds.append(a.get("iscrowd",0))
        boxes = torch.as_tensor(boxes, dtype=torch.float32).reshape(-1,4)
        labels = torch.as_tensor(labels, dtype=torch.int64)
        target = {"boxes": boxes, "labels": labels,
                  "image_id": torch.tensor([image_id]),
                  "area": torch.as_tensor(areas, dtype=torch.float32),
                  "iscrowd": torch.as_tensor(crowds, dtype=torch.int64)}
        # Aumento horizontal sencillo, aplicado de forma sincronizada con las cajas.
        if self.train and random.random() < 0.5:
            image = TF.hflip(image)
            if len(boxes):
                boxes = boxes.clone()
                boxes[:,[0,2]] = image.width - boxes[:,[2,0]]
                target["boxes"] = boxes
        return TF.to_tensor(image), target

def collate_fn(batch): return tuple(zip(*batch))
train_ds = CocoDetectionTorchvision(ANN_FILES["train"], IMAGE_DIRS["train"], train=True)
val_ds = CocoDetectionTorchvision(ANN_FILES["val"], IMAGE_DIRS["val"], train=False)
test_ds = CocoDetectionTorchvision(ANN_FILES["test"], IMAGE_DIRS["test"], train=False) if "test" in ANN_FILES and "test" in IMAGE_DIRS else None
train_loader = DataLoader(train_ds, batch_size=4, shuffle=True, num_workers=2, collate_fn=collate_fn)
val_loader = DataLoader(val_ds, batch_size=2, shuffle=False, num_workers=2, collate_fn=collate_fn)
print("Tamaños:", len(train_ds), len(val_ds), len(test_ds) if test_ds else "test no detectado")

## 5. Entrenar SSD300-VGG16 (baseline)

Se cargan los pesos COCO y se sustituye la cabeza clasificadora por 5 clases + background. Se guarda el mejor modelo según la pérdida de validación. Ajusta épocas y batch size según la GPU disponible.

In [ ]:
from torchvision.models.detection import ssd300_vgg16
from torchvision.models.detection import SSD300_VGG16_Weights

def build_ssd(pretrained=True):
    weights = SSD300_VGG16_Weights.COCO_V1 if pretrained else None
    model = ssd300_vgg16(weights=weights)
    # SSD usa NUM_CLASSES + 1 salidas: una para background.
    in_channels = model.head.classification_head.module_list[0][0].in_channels
    num_anchors = model.anchor_generator.num_anchors_per_location()
    model.head.classification_head = torchvision.models.detection.ssd.SSDClassificationHead(
        in_channels, num_anchors, NUM_CLASSES + 1
    )
    return model

ssd = build_ssd(pretrained=True).to(DEVICE)

In [ ]:
def train_one_epoch(model, loader, optimizer, device):
    model.train(); losses=[]
    for images, targets in loader:
        images = [im.to(device) for im in images]
        targets = [{k:v.to(device) for k,v in t.items()} for t in targets]
        loss_dict = model(images, targets)
        loss = sum(loss_dict.values())
        optimizer.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 10.0)
        optimizer.step()
        losses.append(float(loss.detach().cpu()))
    return float(np.mean(losses))

@torch.no_grad()
def mean_detection_loss(model, loader, device):
    # Torchvision calcula loss solo en modo train; no actualizar gradientes.
    model.train(); vals=[]
    for images, targets in loader:
        images=[im.to(device) for im in images]
        targets=[{k:v.to(device) for k,v in t.items()} for t in targets]
        ld=model(images,targets); vals.append(float(sum(ld.values()).cpu()))
    model.eval()
    return float(np.mean(vals))

EPOCHS_SSD = 15
optimizer = torch.optim.SGD([p for p in ssd.parameters() if p.requires_grad],
                            lr=0.002, momentum=0.9, weight_decay=0.0005)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1)
history_ssd=[]
best_val=float("inf")
for epoch in range(EPOCHS_SSD):
    tr=train_one_epoch(ssd,train_loader,optimizer,DEVICE)
    vl=mean_detection_loss(ssd,val_loader,DEVICE)
    scheduler.step()
    history_ssd.append({"epoch":epoch+1,"train_loss":tr,"val_loss":vl})
    print(f"SSD epoch {epoch+1}/{EPOCHS_SSD}: train={tr:.4f} val={vl:.4f}")
    if vl < best_val:
        best_val=vl
        torch.save(ssd.state_dict(),"ssd300_baseline_best.pth")
ssd.load_state_dict(torch.load("ssd300_baseline_best.pth",map_location=DEVICE))
ssd.eval()
pd.DataFrame(history_ssd).plot(x="epoch",y=["train_loss","val_loss"],grid=True,title="SSD300 baseline")
plt.show()

## 6. Registrar y entrenar Cascade R-CNN (baseline)

Detectron2 utiliza índices de clases desde 0. `register_coco_instances` lee directamente las anotaciones COCO. Cascade se configura sin máscara, como exige el guion.

In [ ]:
# Esta celda requiere detectron2 instalado correctamente.
from detectron2.data.datasets import register_coco_instances
from detectron2.data import MetadataCatalog, DatasetCatalog
from detectron2.engine import DefaultTrainer, DefaultPredictor
from detectron2.config import get_cfg
from detectron2 import model_zoo
from detectron2.evaluation import COCOEvaluator, inference_on_dataset
from detectron2.data import build_detection_test_loader

for split in ("train","val","test"):
    if split in ANN_FILES and split in IMAGE_DIRS:
        name=f"lettuce_{split}"
        if name in DatasetCatalog.list(): DatasetCatalog.remove(name)
        register_coco_instances(name, {}, str(ANN_FILES[split]), str(IMAGE_DIRS[split]))
        MetadataCatalog.get(name).set(thing_classes=CLASS_NAMES)

CASCADE_CONFIG = "Misc/cascade_mask_rcnn_R_50_FPN_3x.yaml"
cfg = get_cfg()
cfg.merge_from_file(model_zoo.get_config_file(CASCADE_CONFIG))
cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url(CASCADE_CONFIG)
cfg.MODEL.MASK_ON = False
cfg.DATASETS.TRAIN = ("lettuce_train",)
cfg.DATASETS.TEST = ("lettuce_val",)
cfg.DATALOADER.NUM_WORKERS = 2
cfg.MODEL.ROI_HEADS.NUM_CLASSES = NUM_CLASSES
cfg.SOLVER.IMS_PER_BATCH = 2
cfg.SOLVER.BASE_LR = 0.00025
cfg.SOLVER.MAX_ITER = max(500, len(train_ds)//2 * 10)
cfg.SOLVER.STEPS = (int(cfg.SOLVER.MAX_ITER*0.7), int(cfg.SOLVER.MAX_ITER*0.9))
cfg.SOLVER.CHECKPOINT_PERIOD = max(100, cfg.SOLVER.MAX_ITER//3)
cfg.TEST.EVAL_PERIOD = max(100, cfg.SOLVER.MAX_ITER//5)
cfg.OUTPUT_DIR = "./output_cascade_baseline"
os.makedirs(cfg.OUTPUT_DIR,exist_ok=True)
# Evita sobrescribir configuración con valores previos del entorno.
trainer = DefaultTrainer(cfg)
trainer.resume_or_load(resume=False)
trainer.train()

## 7. Evaluación COCO: mAP@0.5 y mAP@0.5:0.95

Se evalúa en validation para comparar modelos y elegir mejoras. El test se reserva para la evaluación final. Detectron2 genera automáticamente métricas COCO; SSD se convierte a predicciones COCO para usar el mismo evaluador.

In [ ]:
def evaluate_detectron2(cfg, split="val"):
    evaluator = COCOEvaluator(f"lettuce_{split}", output_dir=cfg.OUTPUT_DIR)
    loader = build_detection_test_loader(cfg, f"lettuce_{split}")
    return inference_on_dataset(DefaultPredictor(cfg).model, loader, evaluator)

# Si el entrenamiento ha terminado, cargar el último/best checkpoint.
cfg.MODEL.WEIGHTS = os.path.join(cfg.OUTPUT_DIR, "model_final.pth")
cascade_results = evaluate_detectron2(cfg, "val")
print(cascade_results)

In [ ]:
@torch.no_grad()
def torchvision_predictions_to_coco(model, dataset, score_thresh=0.001):
    model.eval(); predictions=[]
    loader=DataLoader(dataset,batch_size=2,shuffle=False,num_workers=2,collate_fn=collate_fn)
    for images, targets in loader:
        outputs=model([im.to(DEVICE) for im in images])
        for target,out in zip(targets,outputs):
            image_id=int(target["image_id"].item())
            for box,score,label in zip(out["boxes"].cpu(),out["scores"].cpu(),out["labels"].cpu()):
                if float(score)<score_thresh: continue
                x1,y1,x2,y2=map(float,box.tolist())
                contig=int(label)-1
                if contig<0 or contig>=NUM_CLASSES: continue
                predictions.append({"image_id":image_id,"category_id":cat_ids[contig],
                                    "bbox":[x1,y1,max(0,x2-x1),max(0,y2-y1)],
                                    "score":float(score)})
    return predictions

def coco_metrics(coco_gt, predictions):
    if not predictions:
        return {"mAP50":0.0,"mAP50_95":0.0}
    coco_dt=coco_gt.loadRes(predictions)
    ev=COCOeval(coco_gt,coco_dt,"bbox")
    ev.evaluate(); ev.accumulate(); ev.summarize()
    return {"mAP50_95":float(ev.stats[0]),"mAP50":float(ev.stats[1])}

coco_val=COCO(str(ANN_FILES["val"]))
ssd_val_preds=torchvision_predictions_to_coco(ssd,val_ds)
ssd_val_metrics=coco_metrics(coco_val,ssd_val_preds)
print("SSD300 validation:",ssd_val_metrics)

## 8. Medir latencia de inferencia

Medir después de un calentamiento y sincronizar CUDA. Se informa de ms/imagen y FPS aproximados. Para una comparación justa, utiliza la misma máquina, resolución y batch size.

In [ ]:
@torch.no_grad()
def measure_latency_torchvision(model, dataset, n=50):
    model.eval()
    samples=[dataset[i][0].to(DEVICE) for i in range(min(n,len(dataset)))]
    for im in samples[:5]: model([im])
    if DEVICE.type=="cuda": torch.cuda.synchronize()
    t0=time.perf_counter()
    for im in samples: model([im])
    if DEVICE.type=="cuda": torch.cuda.synchronize()
    elapsed=time.perf_counter()-t0
    ms=1000*elapsed/len(samples)
    return {"ms_per_image":ms,"fps":1000/ms}

ssd_latency=measure_latency_torchvision(ssd,val_ds)
print("SSD300:",ssd_latency)

## 9. Estrategia de mejora: augmentation + ajuste del optimizador (SSD)

Hipótesis: una inversión horizontal aleatoria puede ayudar a generalizar ante cambios de orientación y AdamW puede acelerar la adaptación de la cabeza. Para atribuir el efecto, compara contra baseline cambiando una variable cada vez. Esta versión activa flip horizontal y AdamW; no mezcles sus resultados con los del baseline.

In [ ]:
# Dataset con flip sincronizado en cajas, ya implementado en train_ds.
ssd_improved=build_ssd(pretrained=True).to(DEVICE)
opt_improved=torch.optim.AdamW(ssd_improved.parameters(),lr=1e-4,weight_decay=1e-4)
EPOCHS_IMPROVED=10
hist_improved=[]
for epoch in range(EPOCHS_IMPROVED):
    tr=train_one_epoch(ssd_improved,train_loader,opt_improved,DEVICE)
    vl=mean_detection_loss(ssd_improved,val_loader,DEVICE)
    hist_improved.append({"epoch":epoch+1,"train_loss":tr,"val_loss":vl})
    print(f"SSD improved {epoch+1}: train={tr:.4f} val={vl:.4f}")
torch.save(ssd_improved.state_dict(),"ssd300_improved.pth")
ssd_improved.eval()
ssd_improved_metrics=coco_metrics(coco_val,torchvision_predictions_to_coco(ssd_improved,val_ds))
print("SSD mejorado:",ssd_improved_metrics)

## 10. Comparación de resultados

Rellena la fila de Cascade con los valores de `cascade_results` (Detectron2 imprime `bbox/AP` y `bbox/AP50`). Añade también las métricas de cada experimento. No inventes valores: deben salir de las ejecuciones reales.

In [ ]:
# Introduce aquí los valores reales de Cascade obtenidos en validation.
cascade_map5095 = np.nan
cascade_map50 = np.nan

results = pd.DataFrame([
    {"Modelo":"Cascade R-CNN baseline","mAP@0.5":cascade_map50,"mAP@0.5:0.95":cascade_map5095,
     "ms/imagen":np.nan},
    {"Modelo":"SSD300 baseline","mAP@0.5":ssd_val_metrics["mAP50"],"mAP@0.5:0.95":ssd_val_metrics["mAP50_95"],
     "ms/imagen":ssd_latency["ms_per_image"]},
    {"Modelo":"SSD300 mejorado","mAP@0.5":ssd_improved_metrics["mAP50"],"mAP@0.5:0.95":ssd_improved_metrics["mAP50_95"],
     "ms/imagen":np.nan},
])
display(results)

## 11. Evaluación final en test (solo cuando hayas elegido configuración)

No uses test para decidir hiperparámetros. Ejecuta esta sección una vez seleccionados los modelos definitivos. La métrica test es la que debes reportar como estimación final.

In [ ]:
if test_ds is not None:
    coco_test=COCO(str(ANN_FILES["test"]))
    test_preds=torchvision_predictions_to_coco(ssd,test_ds)
    test_metrics=coco_metrics(coco_test,test_preds)
    print("SSD300 test:",test_metrics)
else:
    print("No se detectó split test; revisa ANN_FILES e IMAGE_DIRS.")

## 12. Inspección cualitativa de errores

Incluye ejemplos de verdaderos positivos, falsos positivos y falsos negativos. Revisa especialmente confusiones entre etapas de crecimiento y objetos pequeños/solapados.

In [ ]:
@torch.no_grad()
def visualize_ssd_predictions(model,dataset,n=4,threshold=0.35):
    model.eval()
    ids=random.sample(range(len(dataset)),min(n,len(dataset)))
    fig,axes=plt.subplots(1,len(ids),figsize=(5*len(ids),5))
    if len(ids)==1: axes=[axes]
    for ax,i in zip(axes,ids):
        image,target=dataset[i]
        out=model([image.to(DEVICE)])[0]
        ax.imshow(image.permute(1,2,0)); ax.axis("off")
        for box,score,label in zip(out["boxes"].cpu(),out["scores"].cpu(),out["labels"].cpu()):
            if score<threshold: continue
            x1,y1,x2,y2=box.tolist()
            ax.add_patch(plt.Rectangle((x1,y1),x2-x1,y2-y1,fill=False,color="lime",linewidth=1.5))
            ax.text(x1,y1,f"{CLASS_NAMES[int(label)-1]} {float(score):.2f}",color="black",
                    bbox=dict(facecolor="lime",alpha=.7),fontsize=8)
        ax.set_title(f"Índice {i}")
    plt.tight_layout()

visualize_ssd_predictions(ssd,val_ds)

## 13. Conclusiones para la memoria y defensa

Completa con resultados medidos:

1. **Dataset:** número de imágenes por partición, número de instancias por clase, ejemplos de cajas y tamaños de objeto.
2. **Baseline:** mAP@0.5, mAP@0.5:0.95, pérdida y latencia de Cascade R-CNN y SSD300.
3. **Hipótesis de mejora:** explica por qué el cambio propuesto debería ayudar.
4. **Resultados:** compara cada mejora con el baseline en la misma partición de validación.
5. **Análisis cualitativo:** muestra aciertos, falsos positivos, falsos negativos y clases confundidas.
6. **Limitaciones:** tamaño del dataset, desbalanceo, coste computacional y variabilidad entre ejecuciones.

### Reproducibilidad
- Semilla fija: `SEED=42`.
- Guarda versiones de PyTorch/Torchvision, CUDA, Detectron2 y configuración.
- Conserva checkpoints, logs y tabla de resultados.
- Incluye en la entrega el notebook limpio, diapositivas PDF y README con instrucciones.